### 1. Base de dados textuais

**Justificativa e explicação da escolha:** Escolhemos o Project Gutenberg por reunir várias obras de domínio público. A estrutura dos arquivos disponibilizados permite a raspagem automatizada e o processamento completo de textos para tarefas de PLN.

**Adequação dos dados às tarefas de PLN (volume, abrangência, variedade):** As obras foram obtidas diretamente via rota de harvest do Gutenberg (/robot/harvest?filetypes[]=html), extraindo arquivos HTML compactados (.zip) em inglês e identificados pelo ID do livro. O script descompacta os arquivos, extrai o texto limpo, amostra trechos do início (20%), meio (50%) e fim (80%) de cada livro e armazena os dados processados em um arquivo CSV.

**Limitação:** A amostragem baseia-se na captura automatizada da página de harvest de arquivos HTML, sem raspagem dedicada das páginas de metadados individuais (como assuntos detalhados ou classificação da Library of Congress). Além disso, a limpeza do texto depende da presença das marcações de cabeçalho/rodapé padrão do Gutenberg.

**Organização e interpretabilidade da base de dados (dicionário de dados):** Inclui colunas geradas no pré-processamento(`summary_normalized`, `summary_clean`, `tokens`, `tokens_no_punctuation`, `tokens_final`, `text_final`, `tokens_lemmatized`, `text_lemmatized`, `tokens_stemmed`, `text_stemmed`).

## 2. Script Python — scraping, acesso via API, outros métodos

**Funcionamento e reprodutibilidade:** o script funciona e é incremental, evitando recoletar livros e obras já salvos.

**Documentação dos procedimentos realizados:** majoritariamente via comentários no código; existe apenas uma célula markdown (`### pré-processamento`) sinalizando o início dessa etapa — os blocos de lematização e stemming ainda não têm introdução própria em markdown.

**Criatividade / capacidade de resolução de problemas:** deduplicação de obras por título+autor normalizado (`work_key`), com normalização Unicode para lidar com nomes em diferentes idiomas,aplicação de limpeza sobre `summary`.




In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import time

BASE_URL = "https://www.gutenberg.org"

HEADERS = {
    "User-Agent": (
        "FURB-PLN-DataExtraction"
        "(academic project; contact: jectrevisol@furb.br, mrharbs@furb.br, moplgalvao@furb.br)"
    )
}

In [ ]:
def get_soup(url):
    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    return BeautifulSoup(response.text, "html.parser")

def load_existing_data():
    try:
        df = pd.read_csv(
            "gutenberg_books.csv",
            dtype={"book_id": str}
        )

        return df

    except FileNotFoundError:
        return pd.DataFrame()


def get_books_from_category(category_url):

    soup = get_soup(category_url)

    books = []

    for link in soup.find_all("a", href=True):

        href = link["href"]

        match = re.fullmatch(r"/ebooks/(\d+)", href)

        if not match:
            continue

        book_id = match.group(1)

        title = link.get_text(" ", strip=True)

        books.append({
            "book_id": book_id,
            "title": title,
            "ebook_url": urljoin(BASE_URL, href)
        })

    next_url = None

    for link in soup.find_all("a", href=True):

        text = link.get_text(" ", strip=True)

        if text == "Next":
            next_url = urljoin(BASE_URL, link["href"])
            break

    return books, next_url

def get_metadata_table(soup):
    metadata = {}

    for row in soup.select("table tr"):
        cells = row.find_all(["th", "td"])

        if len(cells) < 2:
            continue

        key = cells[0].get_text(" ", strip=True)
        value = cells[1].get_text(" ", strip=True)

        if key == "Subject":
            metadata.setdefault("Subject", []).append(value)
        else:
            metadata[key] = value

    return metadata

def get_summary(soup):
    page_body = soup.find("div", class_="page-body")

    if not page_body:
        return None

    text = page_body.get_text(" ", strip=True)

    text = re.sub(r"^QR code\s*", "", text)

    summary = text.split("Read more", 1)[0]

    summary = re.sub(
        r"\(This is an automatically generated summary\.\)",
        "",
        summary
    )

    return summary.strip()

def normalize_text(text):

    if not text:
        return ""

    text = text.lower()

    import unicodedata

    text = unicodedata.normalize(
        "NFKD",
        text
    )

    text = "".join(
        char
        for char in text
        if not unicodedata.combining(char)
    )

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def create_work_key(title, author):

    normalized_title = normalize_text(
        title
    )

    normalized_author = normalize_text(
        author
    )

    return (
        normalized_title
        + "|"
        + normalized_author
    )


def get_book_metadata(book):

    print(
        f"Coletando: {book['title']}"
    )

    soup = get_soup(
        book["ebook_url"]
    )

    metadata = get_metadata_table(
        soup
    )

    language = metadata.get(
        "Language"
    )

    if language != "English":

        print(
            f"Ignorado: idioma = {language}"
        )

        return None

    title = metadata.get(
        "Title",
        book["title"]
    )

    author = metadata.get(
        "Author"
    )

    data = {

        "book_id": book["book_id"],

        "title": title,

        "author": author,

        "language": language,

        "loc_class": metadata.get(
            "LoC Class"
        ),

        "subjects": metadata.get(
            "Subject",
            []
        ),

        "release_date": metadata.get(
            "Release Date"
        ),

        "ebook_url": book["ebook_url"],

        "summary": get_summary(
            soup
        )
    }

    data["subjects"] = " | ".join(
        data["subjects"]
    )

    data["work_key"] = create_work_key(
        title,
        author
    )

    return data

In [ ]:
def main():

    category_url = (
        "https://www.gutenberg.org/ebooks/bookshelf/645"
    )

    df_existing = load_existing_data()

    existing_ids = set()

    existing_work_keys = set()

    if not df_existing.empty:

        if "book_id" in df_existing.columns:

            existing_ids = set(
                df_existing["book_id"]
                .astype(str)
            )

        if "work_key" in df_existing.columns:

            existing_work_keys = set(
                df_existing["work_key"]
                .dropna()
                .astype(str)
            )

        elif (
            "title" in df_existing.columns
            and "author" in df_existing.columns
        ):

            for _, row in df_existing.iterrows():

                work_key = create_work_key(
                    row["title"],
                    row["author"]
                )

                existing_work_keys.add(
                    work_key
                )

    page_number = 1

    visited_pages = set()

    while category_url:

        # aqui ta limitando a execução as 3 primeiras paginas, se executar denovo do msm jeito que ta, ele nao vai pegar nenhum livro,
        # precisa ir aumentando conforme vai executando, precisa executar em partes para nao estourar o limite de requests do gutenberg
        # próx execução, podemos ir aumentando de 3 em 3, só q se executa ate as 6 paginas, dps tem q esperar um tempo pra executar dnv e tal
        if page_number > 4:
            break

        if category_url in visited_pages:

            print(
                "Página já visitada. "
                "Encerrando paginação."
            )

            break

        visited_pages.add(
            category_url
        )

        print(
            f"\n--- Página {page_number} ---"
        )

        print(
            f"URL: {category_url}"
        )

        try:

            books, next_url = (
                get_books_from_category(
                    category_url
                )
            )

        except requests.RequestException as error:

            print(
                f"Erro ao acessar página: {error}"
            )

            break

        print(
            f"Livros encontrados: {len(books)}"
        )

        for book in books:

            book_id = str(
                book["book_id"]
            )

            if book_id in existing_ids:

                print(
                    f"Já coletado: "
                    f"{book['title']} "
                    f"(ID {book_id})"
                )

                continue

            try:

                data = get_book_metadata(
                    book
                )

                if data is None:

                    existing_ids.add(
                        book_id
                    )

                    time.sleep(5)

                    continue

                work_key = data[
                    "work_key"
                ]

                if work_key in existing_work_keys:

                    print(
                        "Ignorado: obra já "
                        "coletada com outro ID"
                    )

                    existing_ids.add(
                        book_id
                    )

                    time.sleep(5)

                    continue

                new_row = pd.DataFrame(
                    [data]
                )

                df_existing = pd.concat(
                    [
                        df_existing,
                        new_row
                    ],
                    ignore_index=True
                )

                df_existing.to_csv(
                    "gutenberg_books.csv",
                    index=False,
                    encoding="utf-8-sig"
                )

                existing_ids.add(
                    book_id
                )

                existing_work_keys.add(
                    work_key
                )

                print(
                    f"Salvo: {data['title']}"
                )

                time.sleep(5)

            except requests.RequestException as error:

                print(
                    f"Erro ao coletar "
                    f"{book['title']}: {error}"
                )

        category_url = next_url

        page_number += 1

        if category_url:

            time.sleep(5)

    print(
        "\nColeta finalizada."
    )

    print(
        f"Total de livros no CSV: "
        f"{len(df_existing)}"
    )


if __name__ == "__main__":
    main()


--- Página 1 ---
URL: https://www.gutenberg.org/ebooks/bookshelf/639
Livros encontrados: 25
Já coletado: Pride and Prejudice Jane Austen 175901 downloads (ID 1342)
Coletando: Romeo and Juliet William Shakespeare 89335 downloads
Salvo: Romeo and Juliet
Já coletado: A Room with a View E. M. Forster 78526 downloads (ID 2641)
Já coletado: The Secret of Chimneys Agatha Christie 73123 downloads (ID 65238)
Já coletado: The Mysteries of Udolpho Ann Ward Radcliffe 71056 downloads (ID 3268)
Já coletado: The Blue Castle: a novel L. M. Montgomery 67153 downloads (ID 67979)
Já coletado: Jane Eyre: An Autobiography Charlotte Brontë 66416 downloads (ID 1260)
Já coletado: Carmen Prosper Mérimée 64795 downloads (ID 2465)
Já coletado: The String of Pearls; Or, The Barber of Fleet Street. A Domestic Romance. Thomas Peckett Prest and James Malcolm Rymer 56189 downloads (ID 59828)
Já coletado: The Green Mummy Fergus Hume 55959 downloads (ID 2868)
Já coletado: Le Fantôme de l'Opéra (French) Gaston Leroux 5

In [ ]:
df = pd.read_csv("gutenberg_books.csv")
df

,book_id,title,author,language,loc_class,subjects,release_date,ebook_url,summary,work_key,trecho
0,1342,Pride and Prejudice,"Austen, Jane, 1775-1817",English,PR: Language and Literatures: English literature,England -- Fiction | Young women -- Fiction | ...,"Jun 1, 1998",https://www.gutenberg.org/ebooks/1342,"""Pride and Prejudice"" by Jane Austen is a nove...",NaN,"Mrs. Gardiner had seen Pemberley, and known th..."
1,2701,"Moby Dick; Or, The Whale","Melville, Herman, 1819-1891",English,PS: Language and Literatures: American and Can...,Whaling -- Fiction | Sea stories | Psychologic...,"Jul 1, 2001",https://www.gutenberg.org/ebooks/2701,"""Moby Dick; Or, The Whale"" by Herman Melville ...",NaN,The acute policy dictating these movements was...
2,2554,Crime and Punishment,"Dostoyevsky, Fyodor, 1821-1881",English,PG: Language and Literatures: Slavic (includin...,Detective and mystery stories | Psychological ...,"Mar 28, 2006",https://www.gutenberg.org/ebooks/2554,"""Crime and Punishment"" by Fyodor Dostoevsky is...",NaN,These exclamations and remarks checked Raskoln...
3,84,"Frankenstein; or, the modern prometheus","Shelley, Mary Wollstonecraft, 1797-1851",English,PR: Language and Literatures: English literature,Science fiction | Horror tales | Gothic fictio...,"Oct 1, 1993",https://www.gutenberg.org/ebooks/84,"""Frankenstein; Or, The Modern Prometheus"" by M...",NaN,“Such were the events that preyed on the heart...
4,11,Alice's Adventures in Wonderland,"Carroll, Lewis, 1832-1898",English,PZ: Language and Literatures: Juvenile belles ...,Fantasy fiction | Children's stories | Imagina...,"Jun 27, 2008",https://www.gutenberg.org/ebooks/11,"""Alice's Adventures in Wonderland"" by Lewis Ca...",NaN,“What _can_ all that green stuff be?” said Ali...
...,...,...,...,...,...,...,...,...,...,...,...
307,76668,Trial by water,"Wright, Sewell Peaslee, 1897-1970",English,PS: Language and Literatures: American and Can...,Short stories | Triangles (Interpersonal relat...,"Aug 10, 2025",https://www.gutenberg.org/ebooks/76668,"""Trial by water by Sewell Peaslee Wright"" is a...",trial by water|wright sewell peaslee 1897 1970,NaN
308,9473,"The Knights of the Cross, or, Krzyzacy: Histor...","Sienkiewicz, Henryk, 1846-1916",English,PG: Language and Literatures: Slavic (includin...,Teutonic Knights -- History -- Fiction | Polan...,"Dec 1, 2005",https://www.gutenberg.org/ebooks/9473,"""The Knights of the Cross, or, Krzyzacy: Histo...",the knights of the cross or krzyzacy historica...,NaN
309,60316,The Bakhtyār Nāma: A Persian Romance,NaN,English,PK: Language and Literatures: Indo-Iranian lit...,Persian poetry -- Translations into English,"Sep 17, 2019",https://www.gutenberg.org/ebooks/60316,"""The Bakhtyār Nāma: A Persian Romance"" by Will...",the bakhtyar nama a persian romance|,NaN
310,19146,"The Entailed Hat; Or, Patty Cannon's Times","Townsend, George Alfred, 1841-1914",English,PS: Language and Literatures: American and Can...,Kidnapping -- Fiction | Eastern Shore (Md. and...,"Aug 30, 2006",https://www.gutenberg.org/ebooks/19146,"""The Entailed Hat; Or, Patty Cannon's Times"" b...",the entailed hat or patty cannon s times|towns...,NaN


### **pré-processamento**


## 3. Limpeza e preparação dos dados
**Dados foram tokenizados?** Sim, foi utilizado o nltk.word_tokenize` aplicado sobre `summary_clean`, gerando a coluna `tokens`.

**Dados foram normalizados?** Sim, o `normalize_text` converte para minúsculas e colapsa espaços antes da tokenização.

**Remoção de ruídos:** Sim,pelo `remove_noise` elimina URLs, e-mails, a frase padrão "(This is an automatically generated summary.)" e sequências de reticências, além de espaços redundantes.

**Remoção de stopwords e pontuação:** sim, em duas etapas,o`remove_punctuation` usa `str.translate` para eliminar pontuação de cada token (descartando tokens que ficam vazios), e `remove_stopwords` filtra a lista de stopwords em inglês do NLTK. O resultado final vira `tokens_final`/`text_final`.

In [ ]:
import re
import string
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

df = pd.read_csv(
    "gutenberg_books.csv",
    dtype={"book_id": str}
)

print(f"Quantidade de livros: {len(df)}")

def normalize_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    text = text.lower()

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text


df["summary_normalized"] = (
    df["summary"]
    .apply(normalize_text)
)

def remove_noise(text):

    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    text = re.sub(
        r"\(this is an automatically generated summary\.\)",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\.{3,}",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


df["summary_clean"] = (
    df["summary_normalized"]
    .apply(remove_noise)
)

def tokenize_text(text):

    if not text:
        return []

    return word_tokenize(text)


df["tokens"] = (
    df["summary_clean"]
    .apply(tokenize_text)
)

def remove_punctuation(tokens):

    return [
        token.translate(str.maketrans("", "", string.punctuation))
        for token in tokens
        if token.translate(str.maketrans("", "", string.punctuation))
    ]


df["tokens_no_punctuation"] = (
    df["tokens"]
    .apply(remove_punctuation)
)

english_stopwords = set(
    stopwords.words("english")
)


def remove_stopwords(tokens):

    return [
        token
        for token in tokens
        if token not in english_stopwords
    ]


df["tokens_final"] = (
    df["tokens_no_punctuation"]
    .apply(remove_stopwords)
)

def tokens_to_text(tokens):

    return " ".join(tokens)


df["text_final"] = (
    df["tokens_final"]
    .apply(tokens_to_text)
)

print("\nExemplos do pré-processamento:\n")

for i in range(min(5, len(df))):

    print("=" * 80)

    print(f"LIVRO {i + 1}")
    print(f"ID: {df.loc[i, 'book_id']}")
    print(f"Título: {df.loc[i, 'title']}")

    print("\n[ORIGINAL]")
    print(df.loc[i, "summary"])

    print("\n[NORMALIZADO]")
    print(df.loc[i, "summary_normalized"])

    print("\n[SEM RUÍDOS]")
    print(df.loc[i, "summary_clean"])

    print("\n[TOKENS]")
    print(df.loc[i, "tokens"])

    print("\n[SEM PONTUAÇÃO]")
    print(df.loc[i, "tokens_no_punctuation"])

    print("\n[SEM STOPWORDS]")
    print(df.loc[i, "tokens_final"])

    print("\n[TEXTO FINAL]")
    print(df.loc[i, "text_final"])


df.to_csv(
    "gutenberg_books.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 80)
print("PRÉ-PROCESSAMENTO CONCLUÍDO")
print("=" * 80)

print(f"Livros processados: {len(df)}")
print("Arquivo atualizado: gutenberg_books.csv")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Quantidade de livros: 312

Exemplos do pré-processamento:

LIVRO 1
ID: 1342
Título: Pride and Prejudice

[ORIGINAL]
"Pride and Prejudice" by Jane Austen is a novel published in 1813. It follows Elizabeth Bennet, who must learn to see past first impressions and hasty judgments. With five daughters and an estate that can only pass to male heirs, the Bennet family faces financial pressure to marry well. When wealthy Mr. Darcy arrives in their countryside neighborhood, his pride and Elizabeth's prejudice set the stage for misunderstandings, hidden truths, and unexpected revelations about character and love.  ...

[NORMALIZADO]
"pride and prejudice" by jane austen is a novel published in 1813. it follows elizabeth bennet, who must learn to see past first impressions and hasty judgments. with five daughters and an estate that can only pass to male heirs, the bennet family faces financial pressure to marry well. when wealthy mr. darcy arrives in their countryside neighborhood, his pride and e

In [ ]:
import ast

from nltk.stem import WordNetLemmatizer

nltk.download("wordnet")
nltk.download("omw-1.4")

df = pd.read_csv(
    "gutenberg_books.csv",
    dtype={"book_id": str}
)

lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):

    if isinstance(tokens, str):
        try:
            tokens = ast.literal_eval(tokens)
        except (ValueError, SyntaxError):
            return []

    if not isinstance(tokens, list):
        return []

    return [
        lemmatizer.lemmatize(token)
        for token in tokens
    ]


df["tokens_lemmatized"] = (
    df["tokens_final"]
    .apply(lemmatize_tokens)
)

def tokens_to_text(tokens):

    return " ".join(tokens)


df["text_lemmatized"] = (
    df["tokens_lemmatized"]
    .apply(tokens_to_text)
)

print("\nExemplos da lematização:\n")

for i in range(min(5, len(df))):

    print("=" * 80)

    print(f"LIVRO {i + 1}")
    print(f"ID: {df.loc[i, 'book_id']}")
    print(f"Título: {df.loc[i, 'title']}")

    print("\n[TOKENS ORIGINAIS]")
    print(df.loc[i, "tokens_final"])

    print("\n[LEMATIZADOS]")
    print(df.loc[i, "tokens_lemmatized"])

    print("\n[TEXTO LEMATIZADO]")
    print(df.loc[i, "text_lemmatized"])

df.to_csv(
    "gutenberg_books.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 80)
print("LEMATIZAÇÃO CONCLUÍDA")
print("=" * 80)

print(f"Livros processados: {len(df)}")
print("Colunas adicionadas:")
print("- tokens_lemmatized")
print("- text_lemmatized")
print("Arquivo atualizado: gutenberg_books.csv")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!



Exemplos da lematização:

LIVRO 1
ID: 1342
Título: Pride and Prejudice

[TOKENS ORIGINAIS]
['pride', 'prejudice', 'jane', 'austen', 'novel', 'published', '1813', 'follows', 'elizabeth', 'bennet', 'must', 'learn', 'see', 'past', 'first', 'impressions', 'hasty', 'judgments', 'five', 'daughters', 'estate', 'pass', 'male', 'heirs', 'bennet', 'family', 'faces', 'financial', 'pressure', 'marry', 'well', 'wealthy', 'mr', 'darcy', 'arrives', 'countryside', 'neighborhood', 'pride', 'elizabeth', 'prejudice', 'set', 'stage', 'misunderstandings', 'hidden', 'truths', 'unexpected', 'revelations', 'character', 'love']

[LEMATIZADOS]
['pride', 'prejudice', 'jane', 'austen', 'novel', 'published', '1813', 'follows', 'elizabeth', 'bennet', 'must', 'learn', 'see', 'past', 'first', 'impression', 'hasty', 'judgment', 'five', 'daughter', 'estate', 'pas', 'male', 'heir', 'bennet', 'family', 'face', 'financial', 'pressure', 'marry', 'well', 'wealthy', 'mr', 'darcy', 'arrives', 'countryside', 'neighborhood', '

In [ ]:
import ast

from nltk.stem import PorterStemmer

nltk.download("punkt")
nltk.download("punkt_tab")

df = pd.read_csv(
    "gutenberg_books.csv",
    dtype={"book_id": str}
)

stemmer = PorterStemmer()

def stem_tokens(tokens):

    if isinstance(tokens, str):
        try:
            tokens = ast.literal_eval(tokens)
        except (ValueError, SyntaxError):
            return []

    if not isinstance(tokens, list):
        return []

    return [
        stemmer.stem(token)
        for token in tokens
    ]


df["tokens_stemmed"] = (
    df["tokens_final"]
    .apply(stem_tokens)
)

def tokens_to_text(tokens):

    return " ".join(tokens)


df["text_stemmed"] = (
    df["tokens_stemmed"]
    .apply(tokens_to_text)
)

print("\nExemplos do stemming:\n")

for i in range(min(5, len(df))):

    print("=" * 80)

    print(f"LIVRO {i + 1}")
    print(f"ID: {df.loc[i, 'book_id']}")
    print(f"Título: {df.loc[i, 'title']}")

    print("\n[TOKENS ORIGINAIS]")
    print(df.loc[i, "tokens_final"])

    print("\n[STEMMED]")
    print(df.loc[i, "tokens_stemmed"])

    print("\n[TEXTO COM STEMMING]")
    print(df.loc[i, "text_stemmed"])

df.to_csv(
    "gutenberg_books.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n" + "=" * 80)
print("STEMMING CONCLUÍDO")
print("=" * 80)

print(f"Livros processados: {len(df)}")
print("Colunas adicionadas:")
print("- tokens_stemmed")
print("- text_stemmed")
print("Arquivo atualizado: gutenberg_books.csv")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!



Exemplos do stemming:

LIVRO 1
ID: 1342
Título: Pride and Prejudice

[TOKENS ORIGINAIS]
['pride', 'prejudice', 'jane', 'austen', 'novel', 'published', '1813', 'follows', 'elizabeth', 'bennet', 'must', 'learn', 'see', 'past', 'first', 'impressions', 'hasty', 'judgments', 'five', 'daughters', 'estate', 'pass', 'male', 'heirs', 'bennet', 'family', 'faces', 'financial', 'pressure', 'marry', 'well', 'wealthy', 'mr', 'darcy', 'arrives', 'countryside', 'neighborhood', 'pride', 'elizabeth', 'prejudice', 'set', 'stage', 'misunderstandings', 'hidden', 'truths', 'unexpected', 'revelations', 'character', 'love']

[STEMMED]
['pride', 'prejudic', 'jane', 'austen', 'novel', 'publish', '1813', 'follow', 'elizabeth', 'bennet', 'must', 'learn', 'see', 'past', 'first', 'impress', 'hasti', 'judgment', 'five', 'daughter', 'estat', 'pass', 'male', 'heir', 'bennet', 'famili', 'face', 'financi', 'pressur', 'marri', 'well', 'wealthi', 'mr', 'darci', 'arriv', 'countrysid', 'neighborhood', 'pride', 'elizabeth',

## 4. Bonus round

**Stemming e lematização:** ambos implementados.
- **Lematização** (`WordNetLemmatizer`): aplicada sobre `tokens_final`. Como não é informada a classe gramatical (`pos`) de cada palavra, o lematizador trata tudo como substantivo por padrão.
- **Stemming** (`PorterStemmer`): mais agressivo, como esperado do algoritmo, reduz "published"→"publish", mas também distorce nomes próprios, limitação conhecida de stemmers baseados em regras.

**Bases adicionais (comparação entre diferentes fontes ou recortes amostrais):** parcialmente atendido, a base já combina duas bookshelves (645 e 639) no mesmo CSV. Falta uma comparação explícita entre os dois recortes para fechar totalmente esse item.

**Automatização na coleta:** parcial, o script evita reprocessar livros já salvos entre execuções, mas a paginação ainda depende de ajuste manual (`page_number > 4`); não há agendamento automático real.